# Held-out generalisation test

Scores an already-trained checkpoint on a dataset it has **never seen**. There
is no training here, so this runs in minutes.

Same-dataset accuracy mostly measures whether a model memorised one dataset's
build pipeline. This measures whether it learned anything about synthesis.

**Set up the session before running:**

| Setting | Value |
|---|---|
| Accelerator | GPU T4 x2 (or P100 - CPU works too, just slower) |
| Internet | **On** (needed for `git clone`) |
| Input | **Notebook Output** of your completed `kaggle_train` run (this carries `best.pt`) |
| Input | A **third** real/fake dataset that was NOT trained on |

To add the checkpoint: **+ Add Input -> Notebook Output** tab -> pick your
finished training notebook.


## 1. Get the project code

In [ ]:
REPO_URL = "https://github.com/Preet1002/Deepfake_Detection.git"

import os, shutil, subprocess, sys
from pathlib import Path

WORKING = Path("/kaggle/working")
PROJECT = WORKING / "Deepfake_Detection"

os.chdir(WORKING)                   # never delete the directory we stand in
if PROJECT.exists():
    shutil.rmtree(PROJECT)

subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT)], check=True)

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
print("working dir:", os.getcwd())


## 2. Find the checkpoint and the datasets

`TRAINED_ON` lists substrings of the datasets the checkpoint was trained on.
Anything mounted that does *not* match is treated as held out. Get this list
wrong and you will "measure generalisation" on training data, so it is spelled
out explicitly rather than inferred.


In [ ]:
from pathlib import Path

INPUT = Path("/kaggle/input")

# ---------------------------------------------------------------------------
# Substrings identifying every dataset the checkpoint WAS trained on. Anything
# else mounted counts as held out.
TRAINED_ON = ["real-vs-fake", "deepfake-and-real-images"]

CHECKPOINT = None          # None = search the mounts; or paste a path here
# ---------------------------------------------------------------------------

SPLIT_NAMES = {"train", "training", "valid", "validation", "val", "test", "testing"}


def subdirs(path):
    try:
        return [c for c in path.iterdir() if c.is_dir()]
    except (PermissionError, OSError):
        return []


def usable_splits(path):
    """Split folders under `path` that actually contain both real/ and fake/."""
    return [c for c in subdirs(path)
            if c.name.lower() in SPLIT_NAMES
            and {d.name.lower() for d in subdirs(c)} >= {"real", "fake"}]


def find_roots(start, max_depth=8):
    found, frontier = [], [(start, 0)]
    while frontier:
        path, depth = frontier.pop(0)
        if len(usable_splits(path)) >= 2:
            found.append(path)
            continue
        if depth < max_depth:
            frontier += [(c, depth + 1) for c in subdirs(path)
                         if c.name.lower() not in {"real", "fake"}]
    return found


def find_checkpoints(start, max_depth=6):
    """Breadth-first hunt for *.pt, never descending into image folders.

    rglob("*.pt") would walk 70k-file image directories on every mounted
    dataset; skipping real/ and fake/ by name keeps this near-instant.
    """
    found, frontier = [], [(start, 0)]
    while frontier:
        path, depth = frontier.pop(0)
        try:
            entries = list(path.iterdir())
        except (PermissionError, OSError):
            continue
        found += [p for p in entries if p.is_file() and p.suffix == ".pt"]
        if depth < max_depth:
            frontier += [(c, depth + 1) for c in entries
                         if c.is_dir() and c.name.lower() not in {"real", "fake"}]
    return found


if CHECKPOINT is None:
    candidates = find_checkpoints(INPUT)
    if not candidates:
        raise SystemExit(
            "No .pt checkpoint found under /kaggle/input.\n"
            "Add it via '+ Add Input' -> 'Notebook Output' tab -> pick your "
            "finished training notebook.")
    # Prefer best.pt over last.pt when a run saved both.
    candidates.sort(key=lambda p: (p.name != "best.pt", str(p)))
    if len(candidates) > 1:
        print("Checkpoints found:")
        for c in candidates:
            print("   ", c)
    CHECKPOINT = str(candidates[0])

print("checkpoint:", CHECKPOINT)

roots = find_roots(INPUT)
if not roots:
    raise SystemExit("No real/fake dataset found under /kaggle/input.")

held_out = [r for r in roots
            if not any(t.lower() in str(r).lower() for t in TRAINED_ON)]

print("\nDatasets mounted:")
for r in roots:
    tag = "HELD OUT" if r in held_out else "trained on"
    print(f"  [{tag:10s}] {r}")

if not held_out:
    raise SystemExit(
        "\nEvery mounted dataset matches TRAINED_ON, so there is nothing held "
        "out to measure.\nAdd a third dataset via '+ Add Input', or fix the "
        "TRAINED_ON list above if one of these really was not trained on.")


## 3. Score the held-out datasets

The checkpoint carries its own model config, so nothing needs configuring here.


In [ ]:
import src.evaluate

for root in held_out:
    print(f"\n===== HELD OUT: {root} =====")
    for split in ("test", "val"):
        try:
            src.evaluate.main([
                "--checkpoint", CHECKPOINT,
                "--data-root", str(root),
                "--split", split,
                "--batch-size", "256",
                "--out-dir", f"/kaggle/working/holdout/{root.name}",
            ])
            break
        except FileNotFoundError as exc:
            # Mirrors vary: some ship no test split, only train/valid.
            print(f"  no {split} split ({exc})")
    else:
        print(f"  neither test nor val usable under {root}; skipped")


## 4. How to read the number

| held-out AUC | what it means |
|---|---|
| **> 0.85** | genuine generalisation - the headline result of the project |
| **0.70 - 0.85** | partial; the model transfers but leans on some dataset cues |
| **0.55 - 0.70** | weak - barely better than guessing on an unseen generator |
| **~ 0.50** | no transfer at all |
| **< 0.45** | *anti*-correlated: it learned a cue that inverts on this dataset |

Whatever it is, report it. A low held-out AUC alongside high in-domain accuracy
is a real, publishable finding about dataset bias - it is the reason
cross-dataset evaluation exists, and it is far more interesting than a single
in-domain accuracy figure.


In [ ]:
import json
from pathlib import Path

for metrics_file in sorted(Path("/kaggle/working/holdout").rglob("metrics.json")):
    m = json.loads(metrics_file.read_text())
    print(f"{metrics_file.parent.name}:")
    for key in ("accuracy", "auc", "ap", "eer", "precision", "recall", "f1"):
        if key in m:
            print(f"   {key:<10}: {m[key]:.4f}")
    print(f"   {'TN/FP/FN/TP':<10}: {m['tn']}/{m['fp']}/{m['fn']}/{m['tp']}")


## 5. Build a demo pack

Samples test images from every mounted dataset, scores each one with this
checkpoint, and writes a manifest recording what the model actually predicts.

The point is that **you know the verdict before you present it**. At ~95%
accuracy roughly one image in twenty is wrong, and finding that out live in
front of an examiner is the worst possible time. Sampling is random and the
manifest lists every result, including the failures - pick from it deliberately
rather than cherry-picking blind.


In [ ]:
import csv, random, shutil
from pathlib import Path

from src.data.dataset import IMAGE_EXTENSIONS, resolve_class_dir, resolve_split_dir
from src.predict import Detector

PER_GROUP = 6              # images per dataset per class
THRESHOLD = 0.5

detector = Detector(CHECKPOINT)
demo = Path("/kaggle/working/demo")
shutil.rmtree(demo, ignore_errors=True)
rows = []

for root in roots:
    seen = "held-out" if root in held_out else "trained-on"
    try:
        split_dir = resolve_split_dir(root, "test")
    except FileNotFoundError:
        split_dir = resolve_split_dir(root, "val")

    for class_name in ("real", "fake"):
        folder = resolve_class_dir(split_dir, class_name)
        if folder is None:
            continue
        paths = sorted(p for p in folder.iterdir()
                       if p.suffix.lower() in IMAGE_EXTENSIONS)
        # Seeded per group so a re-run reproduces the same pack. Seeding with
        # the path string rather than hash(): str hashing is randomised per
        # process, so hash() would resample on every run.
        picked = random.Random(str(folder)).sample(
            paths, min(PER_GROUP, len(paths)))

        out_dir = demo / f"{root.name}__{seen}" / class_name
        out_dir.mkdir(parents=True, exist_ok=True)
        for path in picked:
            from PIL import Image
            score = detector.predict(Image.open(path).convert("RGB"),
                                     detect_faces=False).fake_probability
            predicted = "fake" if score >= THRESHOLD else "real"
            shutil.copy(path, out_dir / path.name)
            rows.append({
                "dataset": root.name, "seen_in_training": seen,
                "true_label": class_name, "predicted": predicted,
                "p_fake": f"{score:.4f}",
                "correct": "yes" if predicted == class_name else "NO",
                "file": str((out_dir / path.name).relative_to(demo)),
            })

with open(demo / "manifest.csv", "w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)

wrong = [r for r in rows if r["correct"] == "NO"]
print(f"{len(rows)} demo images, {len(rows) - len(wrong)} correct, "
      f"{len(wrong)} wrong\n")
print(f"{'dataset':<28} {'seen':<11} {'truth':<5} {'pred':<5} {'p(fake)':>8}  ok")
for r in rows:
    print(f"{r['dataset'][:27]:<28} {r['seen_in_training']:<11} "
          f"{r['true_label']:<5} {r['predicted']:<5} {r['p_fake']:>8}  "
          f"{r['correct']}")


## 6. Package for download

In [ ]:
import os, shutil
from pathlib import Path

WORKING = Path("/kaggle/working")
os.chdir(WORKING)
shutil.rmtree(WORKING / "Deepfake_Detection", ignore_errors=True)   # re-cloneable

for name in ("holdout", "demo"):
    if (WORKING / name).exists():
        archive = shutil.make_archive(str(WORKING / f"{name}_results"), "zip",
                                      root_dir=str(WORKING), base_dir=name)
        print(f"bundle: {archive} "
              f"({Path(archive).stat().st_size / 1e6:.1f} MB)")
